You can use `GridSearchCV` in `scikit-learn` to find the best imputation strategy along with your model. The trick is to treat the imputer as a step in a pipeline and include its parameters (like strategy) in the grid. Then GridSearchCV can evaluate which strategy leads to the best model performance.


In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.DataFrame({
    'age': [25, 32, None, 40, 28, None],
    'income': [50000, 60000, 55000, None, 70000, 65000],
    'gender': ['female', 'male', None, 'male', None, 'female'],
    'target': [0, 1, 0, 1, 0, 1]
})

df

,age,income,gender,target
0,25.0,50000.0,female,0
1,32.0,60000.0,male,1
2,NaN,55000.0,None,0
3,40.0,NaN,male,1
4,28.0,70000.0,None,0
5,NaN,65000.0,female,1


In [3]:
X = df.drop('target', axis=1)
y = df['target']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(include=['object']).columns

In [ ]:
# Numeric imputer
numeric_transformer = SimpleImputer()

# Categorical imputer + encoder
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer()),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# Full pipeline with model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Grid for numeric and categorical imputation
param_grid = {
    'preprocessor__num__strategy': ['mean', 'median', 'constant'],
    'preprocessor__num__fill_value': [0, -999],
    'preprocessor__cat__imputer__fill_value': ['female', 'Unknown'],
}

# GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=2, scoring='accuracy')
grid_search.fit(X, y)

print("Best imputation strategy:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best imputation strategy: {'preprocessor__cat__imputer__fill_value': 'female', 'preprocessor__num__fill_value': 0, 'preprocessor__num__strategy': 'mean'}
Best CV score: 0.5
